# 🔍 Barcode & QR Code Scanner
> **Karim Akoudad** - 126406 — Computer Vision Project  
> UIR — S8 — 2025/2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akoudad/barcode-qr-scanner/blob/barcode-qr-scanner/qr%20-%20push%20.ipynb)

---
**Pipeline:** Grayscale → Gaussian Blur → Morphological Gradient → Otsu Threshold → Morphological Close → Contour Detection → Decode

## Imports & Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from pyzbar.pyzbar import decode

def show_steps(images, titles, cols=4, figsize=(18, 9)):
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles), 1):
        plt.subplot(rows, cols, i)
        if len(img.shape) == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title, fontsize=12)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

##  Step 1 — Upload Image

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
file_bytes = uploaded[filename]

img_array = np.frombuffer(file_bytes, np.uint8)
img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

if img is None:
    raise ValueError("Could not read the uploaded file. Use PNG/JPG/JPEG.")

original = img.copy()
print(f"Image loaded: {filename} — shape: {original.shape}")

##  Step 2 — Grayscale

In [ ]:
gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(6,4))
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.show()

##  Step 3 — Gaussian Blur

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)
plt.figure(figsize=(6,4))
plt.imshow(blur, cmap="gray")
plt.title("Gaussian Blur (5x5)")
plt.axis("off")
plt.show()

##  Step 4 — Morphological Gradient

In [ ]:
grad_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
morph_grad = cv2.morphologyEx(blur, cv2.MORPH_GRADIENT, grad_kernel)
plt.figure(figsize=(6,4))
plt.imshow(morph_grad, cmap="gray")
plt.title("Morphological Gradient")
plt.axis("off")
plt.show()

##  Step 5 — Otsu Threshold

In [ ]:
_, otsu = cv2.threshold(morph_grad, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
plt.figure(figsize=(6,4))
plt.imshow(otsu, cmap="gray")
plt.title("Otsu Threshold")
plt.axis("off")
plt.show()

##  Step 6 — Morphological Close

In [ ]:
close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
closed = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, close_kernel, iterations=2)
plt.figure(figsize=(6,4))
plt.imshow(closed, cmap="gray")
plt.title("Morphological Close")
plt.axis("off")
plt.show()

##  Step 7 — Contour Detection

In [ ]:
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

blob_mask = np.zeros_like(gray)
contour_view = original.copy()
candidate_boxes = []

for c in sorted(contours, key=cv2.contourArea, reverse=True):
    area = cv2.contourArea(c)
    if area < 300:
        continue
    x, y, w, h = cv2.boundingRect(c)
    if w < 20 or h < 20:
        continue
    candidate_boxes.append((x, y, w, h))
    cv2.drawContours(blob_mask, [c], -1, 255, thickness=cv2.FILLED)
    cv2.rectangle(contour_view, (x, y), (x + w, y + h), (0, 255, 0), 2)

print(f"Candidates found: {len(candidate_boxes)}")
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.imshow(blob_mask, cmap="gray"); plt.title("Blob Mask"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(contour_view, cv2.COLOR_BGR2RGB)); plt.title("Contour View"); plt.axis("off")
plt.tight_layout(); plt.show()

##  Step 8 — Decode

In [ ]:
decoded_view = original.copy()
decoded_items = []
seen = set()

def add_decoded_result(obj, x_offset=0, y_offset=0):
    text = obj.data.decode("utf-8", errors="replace")
    key = (obj.type, text)
    if key in seen:
        return
    seen.add(key)
    x, y, w, h = obj.rect
    x += x_offset
    y += y_offset
    cv2.rectangle(decoded_view, (x, y), (x + w, y + h), (255, 0, 255), 3)
    cv2.putText(decoded_view, f"{obj.type}: {text}", (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    decoded_items.append((obj.type, text))

# Try full image first
for obj in decode(gray):
    add_decoded_result(obj)

# Fallback: try each candidate ROI
if not decoded_items:
    for (x, y, w, h) in candidate_boxes:
        pad = 10
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(gray.shape[1], x + w + pad), min(gray.shape[0], y + h + pad)
        roi = gray[y1:y2, x1:x2]
        for obj in decode(roi):
            add_decoded_result(obj, x_offset=x1, y_offset=y1)

print("Decoded results:")
if decoded_items:
    for typ, txt in decoded_items:
        print(f"  [{typ}] {txt}")
else:
    print("  No barcode / QR code detected.")

## Full Pipeline Overview

In [ ]:
show_steps(
    [original, gray, blur, morph_grad, otsu, closed, blob_mask, decoded_view],
    ["0. Original", "1. Grayscale", "2. Gaussian Blur", "3. Morph. Gradient",
     "4. Otsu Threshold", "5. Morph. Close", "6. Blobs", "7. Decoded"]
)